# Examen ICCD753 Recuperación de Información

## 1. Objetivo

Desarrollar un sistema de recuperación de información sobre un corpus asignado. El sistema deberá indexar los documentos, recuperar y ordenar resultados mediante TF-IDF con similitud coseno y BM25, y exponer la búsqueda como un servicio web publicado en internet.

Se evaluará el servicio enviando consultas desde un cliente (p.ej. Postman) y contrastando los resultados con los juicios de relevancia (*qrels*) del corpus.

## 2. Corpus asignado

Cada estudiante recibirá un conjunto de datos. Todos contienen documentos, consultas y qrels. Se utilizarán los identificadores originales de los documentos para que los resultados puedan evaluarse.

| Corpus | Documentos y consultas | Qrels |
| --- | --- | --- |
| NFCorpus | [BeIR/nfcorpus](https://huggingface.co/datasets/BeIR/nfcorpus) | [BeIR/nfcorpus-qrels](https://huggingface.co/datasets/BeIR/nfcorpus-qrels) |
| SciFact | [BeIR/scifact](https://huggingface.co/datasets/BeIR/scifact) | [BeIR/scifact-qrels](https://huggingface.co/datasets/BeIR/scifact-qrels) |

## 3. Requisitos del sistema

### 3.1. Preparación e indexación

El sistema deberá:

1. Cargar e indexar los documentos del corpus asignado.
2. Aplicar y documentar las decisiones de preprocesamiento: tokenización, normalización, eliminación de stopwords y stemming o lematización.
3. Construir un vocabulario y un índice invertido que permita recuperar documentos.
4. Conservar la relación entre cada documento indexado y su identificador original.

La unidad de evaluación será el documento original, porque los qrels relacionan consultas con identificadores de documentos. No se debe dividir el texto en fragmentos.

### 3.2. Modelos de recuperación

Para cualquier consulta de texto, el sistema deberá producir dos rankings independientes:

- TF-IDF con similitud coseno.
- BM25.

Cada ranking deberá ordenar los documentos por puntuación de relevancia y permitir solicitar los primeros `k` resultados.

### 3.3. Servicio web

El sistema deberá estar publicado en internet y ofrecer, como mínimo, los siguientes endpoints:

**Comprobación de disponibilidad**

```http
GET /health
```

**Búsqueda**

```http
POST /search
Content-Type: application/json
```

Ejemplo de solicitud:

```json
{
  "query": "texto de la consulta",
  "model": "bm25",
  "top_k": 10
}
```

El campo `model` deberá aceptar `bm25` y `tfidf`. Una respuesta tendrá esta estructura:

```json
{
  "model": "bm25",
  "results": [
    {
      "rank": 1,
      "document_id": "identificador_original",
      "score": 8.42,
      "title": "Título, si está disponible",
      "snippet": "Fragmento del documento recuperado"
    }
  ]
}
```

Los resultados deberán estar ordenados de mayor a menor puntuación, no contener documentos duplicados y respetar el valor solicitado de `top_k`. El servicio debe aceptar consultas que el estudiante no conozca de antemano.

## 4. Evaluación de la recuperación

Cada estudiante debe evaluar ambos modelos con consultas de la partición `test` y sus qrels. Presentará, como mínimo:

- Precision@10.
- Recall@10.
- NDCG@10.
- Una comparación de los resultados de TF-IDF y BM25.

En el Jupyter Notebook se deberá indicar qué consultas se utilizaron y permitir reproducir los cálculos. La evaluación del docente podrá utilizar otras consultas del mismo conjunto.

## 5. Prueba del docente

El docente accederá a la URL entregada y realizará solicitudes desde un cliente de web service para comprobar:

1. La disponibilidad del servicio.
2. El funcionamiento de TF-IDF y BM25.
3. El formato, orden y validez de los resultados.
4. La correspondencia de los `document_id` con el corpus y los qrels.
5. La respuesta ante consultas no incluidas en los ejemplos de entrega.

## 6. Entregables

Cada estudiante entregará:

1. La URL pública del servicio web, accesible durante el período de revisión.
2. El código en Jupyter Notebook con los resultados de la ejecución de cada bloque de código, incluyendo los resultados de las métricas de evaluación de los modelos de recuperación.

Se evaluará que ambos modelos funcionen correctamente y que cada estudiante interprete sus resultados.

# Desarrollo del examen
**Corpus asignado:** SciFact

En este notebook se desarrollará un sistema de recuperación de información que indexa los documentos de SciFact y permite comparar los modelos TF-IDF con similitud coseno y BM25. Los resultados se evaluarán con las consultas y los juicios de relevancia de la partición de prueba.

### 1. Carga y exploración del corpus

**Entorno y rutas.** Antes de empezar se importan las librerías, se fijan las carpetas de trabajo (relativas a este cuaderno, para que sea reproducible en otro equipo) y se imprimen las versiones usadas. No se emplea ningún motor de búsqueda ni librería de recuperación (scikit-learn, rank_bm25, Elasticsearch…): el índice invertido, TF-IDF y BM25 se implementan a mano con NumPy.

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*IProgress.*")   # aviso cosmético de tqdm que mostraría rutas locales
import os, re, sys, json, math, time, platform, subprocess, socket
from pathlib import Path

import numpy as np
import pandas as pd
import nltk, requests, huggingface_hub, fastapi, uvicorn

RAIZ = Path.cwd()
DATOS, SERVICIO, RESULTADOS = RAIZ / "datos", RAIZ / "servicio", RAIZ / "resultados"
for carpeta in (DATOS, SERVICIO, RESULTADOS):
    carpeta.mkdir(exist_ok=True)

print("Python", platform.python_version(), "|", platform.system())
for modulo in (np, pd, nltk, requests, huggingface_hub, fastapi, uvicorn):
    print(f"{modulo.__name__:>16} {modulo.__version__}")

Python 3.12.10 | Windows
           numpy 2.5.4
          pandas 3.1.1
            nltk 3.10.4
        requests 2.34.2
 huggingface_hub 2.4.0
         fastapi 0.121.2
         uvicorn 0.54.0


**Carga de los datos.** SciFact se toma de los dos repositorios de Hugging Face que indica el enunciado: `BeIR/scifact` (documentos y consultas, en Parquet) y `BeIR/scifact-qrels` (juicios de relevancia en TSV, con particiones `train` y `test`). La función `asegurar_archivo` usa la copia local de `datos/` si ya existe y, si no, la descarga con `huggingface_hub`; así el cuaderno funciona desde cero con conexión y también sin ella una vez descargados los archivos. Los identificadores se mantienen como texto (`str`) tal como vienen en el corpus, porque los qrels se refieren a ellos y el servicio debe devolverlos sin alterarlos.

In [2]:
from huggingface_hub import hf_hub_download


def asegurar_archivo(repo_id: str, ruta: str) -> Path:
    """Devuelve la ruta local del archivo; lo descarga solo si todavía no existe."""
    carpeta = DATOS / repo_id.replace("/", "__")
    destino = carpeta / ruta
    if not destino.exists():
        hf_hub_download(repo_id, ruta, repo_type="dataset", local_dir=carpeta)
    return destino


corpus = pd.read_parquet(asegurar_archivo("BeIR/scifact", "corpus/corpus-00000-of-00001.parquet"))
consultas = pd.read_parquet(asegurar_archivo("BeIR/scifact", "queries/queries-00000-of-00001.parquet"))
leer_qrels = lambda nombre: pd.read_csv(
    asegurar_archivo("BeIR/scifact-qrels", nombre), sep="\t", dtype={"query-id": str, "corpus-id": str}
)
qrels_train_df, qrels_test_df = leer_qrels("train.tsv"), leer_qrels("test.tsv")

corpus["_id"] = corpus["_id"].astype(str)
consultas["_id"] = consultas["_id"].astype(str)
print("corpus:", corpus.shape, list(corpus.columns))
print("consultas:", consultas.shape, list(consultas.columns))
print("qrels train:", qrels_train_df.shape, "| qrels test:", qrels_test_df.shape)
corpus.head(3)

corpus: (5183, 3) ['_id', 'title', 'text']
consultas: (1109, 3) ['_id', 'title', 'text']
qrels train: (919, 3) | qrels test: (339, 3)


,_id,title,text
0,4983,Microstructural development of human newborn c...,Alterations of the architecture of cerebral wh...
1,5836,Induction of myelodysplasia by myeloid-derived...,Myelodysplastic syndromes (MDS) are age-depend...
2,7912,"BC1 RNA, the transcript from a master gene for...",ID elements are short interspersed elements (S...


**Exploración del corpus y de los juicios de relevancia.** Se comprueba cuántos documentos hay, su longitud (en palabras, antes de cualquier preprocesamiento), si existen documentos vacíos o sin título, y cómo son los qrels de cada partición. También se verifica que todos los identificadores citados en los qrels existen en el corpus: sin eso no se podría evaluar.

In [3]:
def a_dict(df):
    """qrels como {query_id: {doc_id: ganancia}}."""
    d = {}
    for qid, did, score in df[["query-id", "corpus-id", "score"]].itertuples(index=False):
        d.setdefault(qid, {})[did] = int(score)
    return d


qrels_train, qrels_test = a_dict(qrels_train_df), a_dict(qrels_test_df)
titulos = dict(zip(corpus["_id"], corpus["title"].fillna("")))
texto_consulta = dict(zip(consultas["_id"], consultas["text"]))

print(f"Documentos: {len(corpus)} | ids únicos: {corpus['_id'].nunique()}")
corpus["n_palabras"] = (corpus["title"].fillna("") + " " + corpus["text"].fillna("")).str.split().str.len()
print("\nLongitud de documento (título + texto, en palabras):")
print(corpus["n_palabras"].describe().round(1).to_string())
print("\nDocumentos con título vacío:", int((corpus["title"].fillna("").str.strip() == "").sum()),
      "| con texto vacío:", int((corpus["text"].fillna("").str.strip() == "").sum()))

print(f"\nConsultas totales en el archivo: {len(consultas)}")
for nombre, df, qr in (("train", qrels_train_df, qrels_train), ("test", qrels_test_df, qrels_test)):
    por_consulta = pd.Series({q: sum(g > 0 for g in d.values()) for q, d in qr.items()})
    print(f"\n[{nombre}] consultas con qrels: {len(qr)} | juicios: {len(df)} | "
          f"relevantes por consulta (media {por_consulta.mean():.2f}, máx {por_consulta.max()})")
    print("  distribución del score:", df["score"].value_counts().sort_index().to_dict())

ids_corpus = set(corpus["_id"])
assert all(d in ids_corpus for qr in (qrels_train, qrels_test) for ds in qr.values() for d in ds), "qrels con ids fuera del corpus"
assert set(qrels_test) <= set(texto_consulta) and set(qrels_train) <= set(texto_consulta)
assert not set(qrels_train) & set(qrels_test), "train y test comparten consultas"
print("\nOK: todos los document_id de los qrels existen en el corpus y train/test no comparten consultas.")

Documentos: 5183 | ids únicos: 5183

Longitud de documento (título + texto, en palabras):
count    5183.0
mean      214.6
std        87.3
min        33.0
25%       158.0
50%       204.0
75%       260.0
max      1541.0

Documentos con título vacío: 0 | con texto vacío: 0

Consultas totales en el archivo: 1109

[train] consultas con qrels: 809 | juicios: 919 | relevantes por consulta (media 1.14, máx 5)
  distribución del score: {1: 919}

[test] consultas con qrels: 300 | juicios: 339 | relevantes por consulta (media 1.13, máx 5)
  distribución del score: {1: 339}

OK: todos los document_id de los qrels existen en el corpus y train/test no comparten consultas.


In [4]:
doc = corpus.iloc[0]
print("DOCUMENTO", doc["_id"], "|", doc["title"])
print(doc["text"][:350], "...\n")
qid_ej = sorted(qrels_test)[0]
print("CONSULTA", qid_ej, ":", texto_consulta[qid_ej])
for did, gan in qrels_test[qid_ej].items():
    print("  relevante:", did, f"(ganancia {gan})", "-", titulos[did][:80])

DOCUMENTO 4983 | Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.
Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotr ...

CONSULTA 1 : 0-dimensional biomaterials show inductive properties.
  relevante: 31715818 (ganancia 1) - New opportunities: the use of nanotechnologies to manipulate and track stem cell


**Lectura de la exploración.** El enunciado exige evaluar con las consultas de la partición `test`; en este corpus son únicamente las consultas que tienen qrels de `test` (el archivo de consultas contiene además las de `train`, que se usarán solo para elegir parámetros). La unidad de recuperación y de evaluación es el **documento completo** (título + resumen), sin dividirlo en fragmentos, porque los qrels apuntan a identificadores de documentos. Los documentos son resúmenes de artículos científicos y las consultas son afirmaciones (claims) de una frase; esa diferencia de longitud y estilo importa más adelante al interpretar los modelos.

### 2. Preprocesamiento

Las decisiones, aplicadas igual a documentos y consultas:

| Paso | Decisión | Motivo |
| --- | --- | --- |
| Normalización | minúsculas y eliminación de acentos (NFKD → ASCII) | «Naïve» y «naive» deben coincidir; las letras griegas se pierden («TNF-α» → `tnf`), pero el resto del término conserva la señal |
| Tokenización | expresión regular `[a-z0-9]+`; se descartan los tokens de un solo carácter | Separa guiones y signos; conserva números y términos como `brca1` o `il`; los símbolos sueltos no aportan |
| Stopwords | lista inglesa de NLTK (198 palabras en la versión instalada) | El corpus está en inglés; reduce ruido y tamaño del índice |
| Stemming | Porter (NLTK), **a confirmar con `train`** | Reúne variantes morfológicas («inhibition», «inhibited» → `inhibit`; aunque «inhibitor» queda aparte); si mejora o no se decide con datos en la sección 5 |

El código se escribe en `servicio/preprocesamiento.py` con `%%writefile`: ese mismo archivo lo importa este cuaderno para evaluar y lo usa el servicio desplegado, así lo que se mide es exactamente lo que se publica.

In [5]:
import nltk
from nltk.corpus import stopwords

NLTK_DIR = RAIZ / "nltk_data"
nltk.data.path.insert(0, str(NLTK_DIR))
try:
    STOP_LISTA = sorted(stopwords.words("english"))
except LookupError:                       # primera ejecución sin copia local
    nltk.download("stopwords", download_dir=str(NLTK_DIR))
    STOP_LISTA = sorted(stopwords.words("english"))

(SERVICIO / "stopwords_en.txt").write_text("\n".join(STOP_LISTA) + "\n", encoding="utf-8", newline="\n")
print(len(STOP_LISTA), "stopwords guardadas en servicio/stopwords_en.txt:", STOP_LISTA[:12], "…")

198 stopwords guardadas en servicio/stopwords_en.txt: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any'] …


In [6]:
%%writefile servicio/preprocesamiento.py
"""Preprocesamiento de texto compartido por la evaluación y el servicio web."""
import re
import unicodedata
from functools import lru_cache
from pathlib import Path

from nltk.stem import PorterStemmer

TOKEN_RE = re.compile(r"[a-z0-9]+")
_STEMMER = PorterStemmer()


@lru_cache(maxsize=100_000)   # acotada: el servicio es público y no debe crecer sin límite
def _raiz(token: str) -> str:
    return _STEMMER.stem(token)


def cargar_stopwords(ruta) -> frozenset:
    lineas = Path(ruta).read_text(encoding="utf-8").splitlines()
    return frozenset(linea.strip() for linea in lineas if linea.strip())


def tokenizar(texto: str) -> list:
    """Minúsculas, sin acentos, tokens alfanuméricos de al menos 2 caracteres."""
    texto = unicodedata.normalize("NFKD", texto.lower()).encode("ascii", "ignore").decode("ascii")
    return [t for t in TOKEN_RE.findall(texto) if len(t) > 1]


def preprocesar(texto: str, stopwords=frozenset(), stem: bool = True) -> list:
    """Tokeniza, quita stopwords y (opcionalmente) aplica el stemmer de Porter."""
    tokens = [t for t in tokenizar(texto) if t not in stopwords]
    return [_raiz(t) for t in tokens] if stem else tokens

Overwriting servicio/preprocesamiento.py

In [7]:
sys.path.insert(0, str(SERVICIO))
from preprocesamiento import preprocesar, cargar_stopwords, tokenizar

STOP = cargar_stopwords(SERVICIO / "stopwords_en.txt")
frase = "Inhibition of BRCA1 and IL-6 signalling does NOT reduce tumour growth in naïve mice."
print("Original      :", frase)
print("Tokens        :", tokenizar(frase))
print("Sin stopwords :", preprocesar(frase, STOP, stem=False))
print("Con Porter    :", preprocesar(frase, STOP, stem=True))

Original      : Inhibition of BRCA1 and IL-6 signalling does NOT reduce tumour growth in naïve mice.
Tokens        : ['inhibition', 'of', 'brca1', 'and', 'il', 'signalling', 'does', 'not', 'reduce', 'tumour', 'growth', 'in', 'naive', 'mice']
Sin stopwords : ['inhibition', 'brca1', 'il', 'signalling', 'reduce', 'tumour', 'growth', 'naive', 'mice']
Con Porter    : ['inhibit', 'brca1', 'il', 'signal', 'reduc', 'tumour', 'growth', 'naiv', 'mice']


### 3. Vocabulario e índice invertido

El **índice invertido** asocia cada término del vocabulario con su lista de documentos (*postings*) y la frecuencia del término en cada uno. Se guarda en formato compacto de NumPy (estilo CSR): un arreglo con todos los documentos de todos los postings concatenados, otro con las frecuencias y un arreglo `offsets` que marca dónde empieza y termina cada término. Además se conservan: `df` (documentos por término), la longitud de cada documento, la longitud media y `N`. La posición interna de cada documento (0…N-1) se traduce al **identificador original** con el arreglo `doc_ids`, de modo que el índice nunca pierde la relación con el corpus ni con los qrels. El índice se puede guardar y cargar con `np.savez_compressed` sin `pickle`.

In [8]:
%%writefile servicio/indice.py
"""Índice invertido propio (formato CSR de NumPy)."""
import json
from collections import Counter

import numpy as np

FORMATO = 1


class IndiceInvertido:
    def __init__(self, doc_ids, terminos, offsets, post_docs, post_tf, doc_len, meta=None):
        self.doc_ids = list(doc_ids)              # posición interna -> id original
        self.terminos = list(terminos)            # id de término -> texto
        self.vocab = {t: i for i, t in enumerate(self.terminos)}
        self.offsets = offsets                    # postings del término i: [offsets[i], offsets[i+1])
        self.post_docs = post_docs                # posiciones internas de documento
        self.post_tf = post_tf                    # frecuencia del término en ese documento
        self.doc_len = doc_len
        self.meta = dict(meta or {})
        self.df = np.diff(offsets).astype(np.int64)
        self.n_docs = len(self.doc_ids)
        self.avgdl = float(doc_len.mean()) if self.n_docs else 0.0

    @classmethod
    def construir(cls, doc_ids, docs_tokens, meta=None):
        vocab, tids, dids, tfs = {}, [], [], []
        doc_len = np.zeros(len(doc_ids), dtype=np.int32)
        for d, tokens in enumerate(docs_tokens):
            doc_len[d] = len(tokens)
            for termino, f in Counter(tokens).items():
                tids.append(vocab.setdefault(termino, len(vocab)))
                dids.append(d)
                tfs.append(f)
        tids = np.asarray(tids, dtype=np.int64)
        orden = np.argsort(tids, kind="stable")      # agrupa por término; dentro, documentos ascendentes
        df = np.bincount(tids, minlength=len(vocab))
        offsets = np.concatenate(([0], np.cumsum(df))).astype(np.int64)
        terminos = [t for t, _ in sorted(vocab.items(), key=lambda kv: kv[1])]
        return cls(doc_ids, terminos, offsets,
                   np.asarray(dids, dtype=np.int32)[orden], np.asarray(tfs, dtype=np.int32)[orden],
                   doc_len, meta)

    def postings(self, termino):
        tid = self.vocab.get(termino)
        if tid is None:
            return np.empty(0, np.int32), np.empty(0, np.int32)
        s, e = self.offsets[tid], self.offsets[tid + 1]
        return self.post_docs[s:e], self.post_tf[s:e]

    def memoria_bytes(self):
        return sum(a.nbytes for a in (self.offsets, self.post_docs, self.post_tf, self.doc_len))

    def guardar(self, ruta):
        meta = {**self.meta, "formato": FORMATO, "n_docs": self.n_docs}
        np.savez_compressed(ruta, doc_ids=np.array(self.doc_ids, dtype=str),
                            terminos=np.array(self.terminos, dtype=str), offsets=self.offsets,
                            post_docs=self.post_docs, post_tf=self.post_tf, doc_len=self.doc_len,
                            meta=np.array(json.dumps(meta)))

    @classmethod
    def cargar(cls, ruta):
        z = np.load(ruta, allow_pickle=False)
        meta = json.loads(str(z["meta"]))
        if meta.get("formato") != FORMATO:
            raise ValueError("Formato de índice no soportado")
        return cls(z["doc_ids"].tolist(), z["terminos"].tolist(), z["offsets"], z["post_docs"],
                   z["post_tf"], z["doc_len"], meta)

Overwriting servicio/indice.py


In [9]:
from indice import IndiceInvertido

ids_docs = corpus["_id"].tolist()
textos_docs = (corpus["title"].fillna("") + " " + corpus["text"].fillna("")).tolist()   # documento = título + texto

t0 = time.perf_counter()
tokens_stem = [preprocesar(t, STOP, stem=True) for t in textos_docs]
indice = IndiceInvertido.construir(ids_docs, tokens_stem, {"stem": True})
print(f"Índice construido en {time.perf_counter() - t0:.1f} s")
print(f"N = {indice.n_docs} documentos | vocabulario = {len(indice.terminos):,} términos | "
      f"postings = {len(indice.post_docs):,} | longitud media = {indice.avgdl:.1f} tokens")
print(f"Memoria de los arreglos del índice: {indice.memoria_bytes() / 1e6:.1f} MB")

por_df = np.argsort(-indice.df)[:10]
print("\nTérminos con mayor df:", [(indice.terminos[i], int(indice.df[i])) for i in por_df])
print("Términos que aparecen en un solo documento:", f"{int((indice.df == 1).sum()):,}",
      f"({(indice.df == 1).mean():.0%} del vocabulario)")

Índice construido en 1.2 s
N = 5183 documentos | vocabulario = 26,636 términos | postings = 455,806 | longitud media = 144.7 tokens
Memoria de los arreglos del índice: 3.9 MB

Términos con mayor df: [('cell', 2529), ('result', 2372), ('studi', 2213), ('use', 2182), ('activ', 1833), ('show', 1578), ('associ', 1536), ('increas', 1529), ('express', 1526), ('function', 1507)]
Términos que aparecen en un solo documento: 13,538 (51% del vocabulario)


**Comprobaciones del índice.** (1) Para un término de ejemplo, sus postings deben coincidir con una búsqueda por fuerza bruta sobre los documentos tokenizados; (2) la relación posición interna ↔ id original se conserva; (3) guardar y volver a cargar el índice no cambia nada.

In [10]:
termino = "insulin"
docs_pos, tfs = indice.postings(termino)
fuerza_bruta = {d: toks.count(termino) for d, toks in enumerate(tokens_stem) if termino in toks}
assert dict(zip(docs_pos.tolist(), tfs.tolist())) == fuerza_bruta
print(f"'{termino}': df = {indice.df[indice.vocab[termino]]}; primeros postings (id original, tf):",
      [(indice.doc_ids[d], int(f)) for d, f in list(zip(docs_pos, tfs))[:4]])
assert indice.doc_ids == ids_docs and indice.postings("zzzz-no-existe")[0].size == 0

ruta_indice = DATOS / "indice_scifact_porter.npz"
indice.guardar(ruta_indice)
recargado = IndiceInvertido.cargar(ruta_indice)
assert recargado.doc_ids == indice.doc_ids and recargado.terminos == indice.terminos
assert all(np.array_equal(getattr(recargado, a), getattr(indice, a)) for a in ("offsets", "post_docs", "post_tf", "doc_len"))
print(f"Índice guardado y recargado sin pérdidas ({ruta_indice.stat().st_size / 1e6:.1f} MB en disco).")

'insulin': df = 162; primeros postings (id original, tf): [('120626', 5), ('195352', 7), ('712078', 1), ('750781', 1)]


Índice guardado y recargado sin pérdidas (1.3 MB en disco).


### 4. Modelos de recuperación

Ambos modelos puntúan un documento sumando contribuciones de los términos de la consulta que están en el vocabulario. Se precalcula el peso de cada *posting*, de modo que buscar es acumular `peso × peso de consulta` sobre unos pocos postings.

**TF-IDF con similitud coseno.** Peso de un término en un documento: `(1 + ln tf) · idf`, con `idf = ln(N / df)`. La consulta usa el mismo esquema (con su propio `tf`). El coseno divide por la norma L2 del documento (calculada sobre todos sus términos) y por la norma de la consulta, por lo que el valor está en [0, 1].

**BM25** (variante Robertson/Lucene). Contribución de cada término de la consulta:
`idf · tf·(k1+1) / (tf + k1·(1 − b + b·dl/avgdl))`, con `idf = ln(1 + (N − df + 0.5)/(df + 0.5))` (siempre positivo). `k1` controla la saturación de la frecuencia y `b` la normalización por longitud; se eligen con `train` en la sección 5. Si un término se repite en la consulta, cada aparición suma.

**Rankings.** Orden por puntuación descendente y, en empates, por `document_id` ascendente (determinista). Sin duplicados. Se devuelven exactamente `min(top_k, N)` documentos: si hay menos documentos con puntuación mayor que 0 que `top_k`, se completa con documentos de puntuación 0 en orden de id. Una consulta vacía, o formada solo por stopwords, lanza `ConsultaInvalida`.

In [11]:
%%writefile servicio/modelos.py
"""TF-IDF (coseno) y BM25 sobre el índice invertido."""
import json
import math
from collections import Counter
from pathlib import Path

import numpy as np

from indice import IndiceInvertido
from preprocesamiento import preprocesar

MODELOS = ("bm25", "tfidf")


class ConsultaInvalida(ValueError):
    """La consulta no contiene ningún término indexable."""


def fragmento(texto: str, n: int = 300) -> str:
    texto = " ".join((texto or "").split())
    if len(texto) <= n:
        return texto
    corte = texto[:n].rsplit(" ", 1)[0] or texto[:n]
    return corte.rstrip(" ,;:.") + "…"


class Buscador:
    def __init__(self, indice, stopwords, stem=True, k1=1.5, b=0.75, documentos=None):
        self.indice, self.stopwords, self.stem = indice, stopwords, stem
        self.documentos = documentos or {}            # id -> (título, texto)
        n, v = indice.n_docs, len(indice.terminos)
        self._termino_de_posting = np.repeat(np.arange(v), indice.df)
        # TF-IDF: peso (1+ln tf)·idf normalizado por la norma L2 del documento
        self.idf_tfidf = np.log(n / indice.df)
        w = (1.0 + np.log(indice.post_tf)) * self.idf_tfidf[self._termino_de_posting]
        norma = np.sqrt(np.bincount(indice.post_docs, weights=w ** 2, minlength=n))
        norma[norma == 0] = 1.0
        self._w_tfidf = w / norma[indice.post_docs]
        # BM25
        self.idf_bm25 = np.log(1.0 + (n - indice.df + 0.5) / (indice.df + 0.5))
        self.fijar_bm25(k1, b)
        # rango de cada documento por id (desempates deterministas)
        orden = np.argsort(np.array(indice.doc_ids, dtype=str), kind="stable")
        self._rango_id = np.empty(n, dtype=np.int64)
        self._rango_id[orden] = np.arange(n)

    def fijar_bm25(self, k1, b):
        ind = self.indice
        self.k1, self.b = k1, b
        dl = ind.doc_len[ind.post_docs]
        tf = ind.post_tf
        sat = tf * (k1 + 1.0) / (tf + k1 * (1.0 - b + b * dl / ind.avgdl))
        self._w_bm25 = self.idf_bm25[self._termino_de_posting] * sat

    @classmethod
    def desde_documentos(cls, documentos, stopwords, stem=True, k1=1.5, b=0.75):
        """documentos: lista de dicts con _id, title y text (documento indexado = título + texto)."""
        ids = [str(d["_id"]) for d in documentos]
        tokens = [preprocesar(f"{d.get('title') or ''} {d.get('text') or ''}", stopwords, stem) for d in documentos]
        indice = IndiceInvertido.construir(ids, tokens, {"stem": stem})
        textos = {str(d["_id"]): (d.get("title") or "", d.get("text") or "") for d in documentos}
        return cls(indice, stopwords, stem, k1, b, textos)

    @classmethod
    def desde_jsonl(cls, ruta, stopwords, stem=True, k1=1.5, b=0.75):
        with open(ruta, encoding="utf-8") as f:
            docs = [json.loads(linea) for linea in f if linea.strip()]
        return cls.desde_documentos(docs, stopwords, stem, k1, b)

    def _puntuar(self, tokens, modelo):
        ind = self.indice
        cuenta = Counter(t for t in tokens if t in ind.vocab)
        acc = np.zeros(ind.n_docs)
        if modelo == "bm25":
            for t, c in cuenta.items():
                tid = ind.vocab[t]
                s, e = ind.offsets[tid], ind.offsets[tid + 1]
                acc[ind.post_docs[s:e]] += c * self._w_bm25[s:e]
        else:
            pesos = {ind.vocab[t]: (1.0 + math.log(c)) * self.idf_tfidf[ind.vocab[t]] for t, c in cuenta.items()}
            norma_q = math.sqrt(sum(p * p for p in pesos.values()))
            if norma_q > 0:
                for tid, wq in pesos.items():
                    s, e = ind.offsets[tid], ind.offsets[tid + 1]
                    acc[ind.post_docs[s:e]] += wq * self._w_tfidf[s:e]
                acc = np.minimum(acc / norma_q, 1.0)
        return acc

    def buscar(self, consulta, modelo="bm25", top_k=10):
        """Devuelve [(document_id, score)] ordenado por score desc. y document_id asc."""
        if modelo not in MODELOS:
            raise ValueError(f"Modelo desconocido: {modelo!r}")
        if isinstance(top_k, bool) or not isinstance(top_k, int) or top_k < 1:
            raise ValueError("top_k debe ser un entero positivo")
        tokens = preprocesar(consulta, self.stopwords, self.stem)
        if not tokens:
            raise ConsultaInvalida("La consulta no contiene términos indexables (vacía o solo stopwords).")
        acc = self._puntuar(tokens, modelo)
        orden = np.lexsort((self._rango_id, -acc))[: min(top_k, self.indice.n_docs)]
        return [(self.indice.doc_ids[i], float(acc[i])) for i in orden]

Overwriting servicio/modelos.py


In [12]:
from modelos import Buscador, ConsultaInvalida, MODELOS

# Verificación a mano con un corpus de 3 documentos (sin stopwords ni stemming)
mini = [{"_id": "d1", "title": "", "text": "apple banana apple"},
        {"_id": "d2", "title": "", "text": "banana cherry"},
        {"_id": "d3", "title": "", "text": "apple cherry cherry cherry"}]
b_mini = Buscador.desde_documentos(mini, frozenset(), stem=False, k1=1.5, b=0.75)

N, df_apple, avgdl = 3, 2, (3 + 2 + 4) / 3
# BM25 de la consulta "apple": d1 (tf=2, dl=3), d3 (tf=1, dl=4), d2 no contiene el término
idf = math.log(1 + (N - df_apple + 0.5) / (df_apple + 0.5))
bm25_d1 = idf * 2 * 2.5 / (2 + 1.5 * (1 - 0.75 + 0.75 * 3 / avgdl))
bm25_d3 = idf * 1 * 2.5 / (1 + 1.5 * (1 - 0.75 + 0.75 * 4 / avgdl))
res = dict(b_mini.buscar("apple", "bm25", 3))
assert math.isclose(res["d1"], bm25_d1) and math.isclose(res["d3"], bm25_d3) and res["d2"] == 0.0
print("BM25  a mano == implementación:", round(bm25_d1, 6), round(bm25_d3, 6), "y 0 para d2")

# TF-IDF coseno de "apple" contra d1: todos los términos tienen idf = ln(3/2)
idf_t = math.log(3 / 2)
w_apple, w_banana = (1 + math.log(2)) * idf_t, 1.0 * idf_t
coseno_d1 = (w_apple * idf_t) / (math.sqrt(w_apple ** 2 + w_banana ** 2) * idf_t)
res = dict(b_mini.buscar("apple", "tfidf", 3))
assert math.isclose(res["d1"], coseno_d1) and all(0.0 <= s <= 1.0 for s in res.values())
print("TF-IDF a mano == implementación:", round(coseno_d1, 6))

BM25  a mano == implementación: 0.671434 0.408699 y 0 para d2
TF-IDF a mano == implementación: 0.861037


**Pruebas de comportamiento sobre el corpus real** (índice con Porter): consulta conocida, orden, ausencia de duplicados, `top_k` exacto, desempate, consulta vacía o solo de stopwords, coseno acotado y término fuera del vocabulario.

In [13]:
buscador_prueba = Buscador(indice, STOP, stem=True, k1=1.5, b=0.75,
                           documentos={d: (titulos[d], "") for d in ids_docs})
for modelo in MODELOS:
    for k in (1, 10, 100):
        r = buscador_prueba.buscar("insulin resistance and type 2 diabetes", modelo, k)
        ids_r, sc = [d for d, _ in r], [s for _, s in r]
        assert len(r) == k and len(set(ids_r)) == k
        assert all(a >= b for a, b in zip(sc, sc[1:]))
        assert all(a > b or sorted(ids_r[i:i + 2]) == ids_r[i:i + 2] for i, (a, b) in enumerate(zip(sc, sc[1:])) if a == b)
    assert max(s for _, s in buscador_prueba.buscar("insulin resistance", modelo, 5)) > 0
assert all(s <= 1.0 for _, s in buscador_prueba.buscar("insulin resistance", "tfidf", 50))
assert len(buscador_prueba.buscar("insulin", "bm25", indice.n_docs + 500)) == indice.n_docs   # k > N -> N
for vacia in ("", "   ", "the of and", "¿?"):
    try:
        buscador_prueba.buscar(vacia, "bm25", 5)
        raise AssertionError("debía rechazar la consulta vacía")
    except ConsultaInvalida:
        pass
r0 = buscador_prueba.buscar("qwertyzzz plokijuh", "bm25", 5)            # términos desconocidos
assert len(r0) == 5 and all(s == 0.0 for _, s in r0) and [d for d, _ in r0] == sorted(ids_docs)[:5]
print("Todas las pruebas de los modelos pasan.")
print("Ejemplo BM25:", [(d, round(s, 3), titulos[d][:40]) for d, s in buscador_prueba.buscar("insulin resistance and type 2 diabetes", "bm25", 3)])

Todas las pruebas de los modelos pasan.
Ejemplo BM25: [('120626', 22.031, 'Mechanisms linking obesity to insulin re'), ('195352', 21.886, 'Insulin action and resistance in obesity'), ('27166444', 21.359, 'Preservation of pancreatic beta-cell fun')]


### 5. Elección de parámetros con `train` (sin usar `test`)

Para no ajustar el sistema sobre las consultas con que luego se evalúa, las decisiones que quedaban abiertas se toman con las consultas de la partición `train` y sus qrels:
* ¿Con o sin stemming de Porter?
* `k1` ∈ {0.9, 1.2, 1.5, 2.0} y `b` ∈ {0.5, 0.75, 0.9} para BM25.

Criterio: mayor NDCG@10 medio en `train`. Primero se definen las métricas, implementadas a mano:
* **Precision@10** = relevantes recuperados en el top 10 / 10.
* **Recall@10** = relevantes recuperados en el top 10 / total de relevantes de la consulta.
* **NDCG@10** = DCG@10 / DCG ideal, con `DCG = Σ ganancia_i / log2(i + 1)` (posición `i` desde 1). Es un documento «relevante» si su ganancia en los qrels es mayor que 0.

In [14]:
def precision_at_k(ranking, relevantes, k=10):
    return sum(d in relevantes for d in ranking[:k]) / k


def recall_at_k(ranking, relevantes, k=10):
    return sum(d in relevantes for d in ranking[:k]) / len(relevantes)


def ndcg_at_k(ranking, ganancias, k=10):
    dcg = sum(ganancias.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranking[:k]))
    ideal = sorted(ganancias.values(), reverse=True)[:k]
    idcg = sum(g / math.log2(i + 2) for i, g in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


# Casos pequeños verificables a mano
rank = ["a", "b", "c", "d"]
assert math.isclose(precision_at_k(rank, {"a", "c", "e"}, 3), 2 / 3)
assert math.isclose(recall_at_k(rank, {"a", "c", "e"}, 3), 2 / 3)
# ganancias {a:1, c:1, e:1}; ranking a,b,c -> DCG = 1 + 0 + 1/log2(4) = 1.5; ideal = 1 + 1/log2(3) + 1/2
assert math.isclose(ndcg_at_k(rank, {"a": 1, "c": 1, "e": 1}, 3), 1.5 / (1 + 1 / math.log2(3) + 0.5))
# ganancias graduadas {a:2, b:1}; ranking b,a -> DCG = 1 + 2/log2(3); ideal = 2 + 1/log2(3)
assert math.isclose(ndcg_at_k(["b", "a"], {"a": 2, "b": 1}, 10), (1 + 2 / math.log2(3)) / (2 + 1 / math.log2(3)))
assert ndcg_at_k(["a", "b"], {"a": 1}, 10) == 1.0 and ndcg_at_k(["x", "y"], {"a": 1}, 10) == 0.0
print("Métricas verificadas a mano.")


def evaluar(buscador, modelo, qrels, k=10):
    """Una fila por consulta con P@k, R@k y NDCG@k."""
    filas = []
    for qid in sorted(qrels):
        ganancias = {d: g for d, g in qrels[qid].items() if g > 0}
        ranking = [d for d, _ in buscador.buscar(texto_consulta[qid], modelo, k)]
        filas.append({"query_id": qid, "modelo": modelo,
                      "p_at_10": precision_at_k(ranking, set(ganancias), k),
                      "r_at_10": recall_at_k(ranking, set(ganancias), k),
                      "ndcg_at_10": ndcg_at_k(ranking, ganancias, k)})
    return pd.DataFrame(filas)

Métricas verificadas a mano.


In [15]:
documentos = corpus[["_id", "title", "text"]].to_dict("records")
filas, buscadores = [], {}
t0 = time.perf_counter()
for stem in (True, False):
    bus = buscadores[stem] = Buscador.desde_documentos(documentos, STOP, stem=stem)
    filas.append({"stemming": stem, "modelo": "tfidf", "k1": None, "b": None,
                  "ndcg_train": evaluar(bus, "tfidf", qrels_train)["ndcg_at_10"].mean()})
    for k1 in (0.9, 1.2, 1.5, 2.0):
        for b in (0.5, 0.75, 0.9):
            bus.fijar_bm25(k1, b)
            filas.append({"stemming": stem, "modelo": "bm25", "k1": k1, "b": b,
                          "ndcg_train": evaluar(bus, "bm25", qrels_train)["ndcg_at_10"].mean()})
print(f"{len(qrels_train)} consultas de train evaluadas en {time.perf_counter() - t0:.0f} s")
seleccion = pd.DataFrame(filas)
bm = seleccion[seleccion["modelo"] == "bm25"]
print("\nBM25 — NDCG@10 en train (filas: k1, columnas: b)")
for stem in (True, False):
    tabla = bm[bm["stemming"] == stem].pivot(index="k1", columns="b", values="ndcg_train").round(4)
    print(f"\nstemming = {stem}\n{tabla.to_string()}")
print("\nTF-IDF — NDCG@10 en train:", seleccion[seleccion["modelo"] == "tfidf"][["stemming", "ndcg_train"]].round(4).to_dict("records"))

mejor = bm.sort_values(["ndcg_train", "stemming", "k1", "b"], ascending=[False, False, True, True]).iloc[0]
CONFIG = {"stem": bool(mejor["stemming"]), "k1": float(mejor["k1"]), "b": float(mejor["b"])}
print("\nConfiguración elegida (solo con train):", CONFIG, f"-> NDCG@10 train = {mejor['ndcg_train']:.4f}")
(SERVICIO / "config.json").write_text(json.dumps(CONFIG, indent=2) + "\n", encoding="utf-8", newline="\n")

809 consultas de train evaluadas en 8 s

BM25 — NDCG@10 en train (filas: k1, columnas: b)

stemming = True
b      0.50    0.75    0.90
k1                         
0.9  0.6936  0.6941  0.6966
1.2  0.6958  0.6972  0.6958
1.5  0.6944  0.6940  0.6932
2.0  0.6903  0.6929  0.6912

stemming = False
b      0.50    0.75    0.90
k1                         
0.9  0.6677  0.6681  0.6657
1.2  0.6669  0.6675  0.6655
1.5  0.6650  0.6658  0.6630
2.0  0.6645  0.6655  0.6595

TF-IDF — NDCG@10 en train: [{'stemming': True, 'ndcg_train': 0.669}, {'stemming': False, 'ndcg_train': 0.6449}]

Configuración elegida (solo con train): {'stem': True, 'k1': 1.2, 'b': 0.75} -> NDCG@10 train = 0.6972


45

**Lectura de la selección (solo `train`, 809 consultas).**

* El **stemming de Porter ayuda**: con él, BM25 llega a 0.6972 NDCG@10 (k1 = 1.2, b = 0.75) frente a 0.6681 como mejor resultado sin stemming (≈ +0.03), y TF-IDF pasa de 0.6449 a 0.6690.
* La **rejilla de BM25 es muy plana**: con stemming, las 12 combinaciones quedan entre 0.6903 y 0.6972, así que `k1` y `b` importan poco en este corpus; el stemming importa bastante más.
* Configuración fijada: **Porter, k1 = 1.2, b = 0.75**; se guarda en `servicio/config.json` y es la que usa el servicio. Con diferencias de milésimas entre combinaciones no se afirma que ese punto sea «el óptimo», solo que es la mejor de la rejilla según un criterio fijado antes de mirar `test`.

### 6. Evaluación en la partición `test`

Se construye el buscador **con exactamente lo que usará el servicio**: `servicio/corpus.jsonl`, `servicio/stopwords_en.txt` y `servicio/config.json`. Las consultas evaluadas son todas las que tienen juicios en `test`; se calculan P@10, R@10 y NDCG@10 por consulta para TF-IDF y BM25 y se promedian. Los qrels de `test` no se usaron para ninguna decisión anterior.

In [16]:
with open(SERVICIO / "corpus.jsonl", "w", encoding="utf-8", newline="\n") as f:
    for d in documentos:
        f.write(json.dumps({"_id": d["_id"], "title": d["title"] or "", "text": d["text"] or ""}, ensure_ascii=False) + "\n")

buscador = Buscador.desde_jsonl(SERVICIO / "corpus.jsonl", cargar_stopwords(SERVICIO / "stopwords_en.txt"), **CONFIG)
qids_test = sorted(qrels_test)
print(f"Consultas de test evaluadas: {len(qids_test)} (de {len(consultas)} consultas del archivo)")
print("Primeros ids:", qids_test[:10], "...")
print("Ejemplo de consulta:", texto_consulta[qids_test[0]])

por_consulta = pd.concat([evaluar(buscador, m, qrels_test) for m in ("tfidf", "bm25")], ignore_index=True)
resumen = (por_consulta.groupby("modelo")[["p_at_10", "r_at_10", "ndcg_at_10"]]
           .agg(["mean", "std"]).round(4))
print("\nResultados en test (media y desviación típica por consulta)")
resumen

Consultas de test evaluadas: 300 (de 1109 consultas del archivo)
Primeros ids: ['1', '100', '1012', '1014', '1019', '1020', '1021', '1024', '1029', '1041'] ...
Ejemplo de consulta: 0-dimensional biomaterials show inductive properties.

Resultados en test (media y desviación típica por consulta)


p_at_10         r_at_10         ndcg_at_10        
          mean     std    mean     std       mean     std
modelo                                                   
bm25    0.0910  0.0506  0.8245  0.3698     0.6816  0.3784
tfidf   0.0903  0.0518  0.8183  0.3746     0.6450  0.3788

In [17]:
medias = por_consulta.groupby("modelo")[["p_at_10", "r_at_10", "ndcg_at_10"]].mean()
print(medias.round(4).rename(columns={"p_at_10": "P@10", "r_at_10": "R@10", "ndcg_at_10": "NDCG@10"}).to_string())

ancho = por_consulta.pivot(index="query_id", columns="modelo", values=["p_at_10", "r_at_10", "ndcg_at_10"])
print("\nComparación consulta a consulta (BM25 frente a TF-IDF):")
rng = np.random.default_rng(42)
comparacion = {}
for nombre, col in (("P@10", "p_at_10"), ("R@10", "r_at_10"), ("NDCG@10", "ndcg_at_10")):
    dif = (ancho[col]["bm25"] - ancho[col]["tfidf"]).to_numpy()
    gana, pierde, empata = int((dif > 1e-12).sum()), int((dif < -1e-12).sum()), int((np.abs(dif) <= 1e-12).sum())
    medias_boot = dif[rng.integers(0, len(dif), (10000, len(dif)))].mean(axis=1)
    ic = np.percentile(medias_boot, [2.5, 97.5])
    n, k = gana + pierde, min(gana, pierde)
    p_signos = min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)
    comparacion[nombre] = {"dif_media": float(dif.mean()), "ic95_bootstrap": [float(ic[0]), float(ic[1])],
                           "bm25_gana": gana, "tfidf_gana": pierde, "empates": empata, "p_prueba_signos": p_signos}
    print(f"  {nombre:8} diferencia media = {dif.mean():+.4f}  IC95% bootstrap [{ic[0]:+.4f}, {ic[1]:+.4f}] | "
          f"BM25 mejor en {gana}, TF-IDF mejor en {pierde}, empates {empata} | prueba de signos p = {p_signos:.2g}")

por_consulta.to_csv(RESULTADOS / "metricas_por_consulta.csv", index=False, float_format="%.6f")
salida = {"corpus": "BeIR/scifact", "particion": "test", "n_consultas": len(qids_test), "configuracion": CONFIG,
          "metricas_medias": medias.round(6).to_dict("index"), "comparacion_bm25_vs_tfidf": comparacion}
(RESULTADOS / "metricas_test.json").write_text(json.dumps(salida, indent=2, ensure_ascii=False) + "\n", encoding="utf-8", newline="\n")
print("\nGuardado: resultados/metricas_test.json y resultados/metricas_por_consulta.csv")

          P@10    R@10  NDCG@10
modelo                         
bm25    0.0910  0.8245   0.6816
tfidf   0.0903  0.8183   0.6450

Comparación consulta a consulta (BM25 frente a TF-IDF):
  P@10     diferencia media = +0.0007  IC95% bootstrap [-0.0037, +0.0047] | BM25 mejor en 19, TF-IDF mejor en 15, empates 266 | prueba de signos p = 0.61
  R@10     diferencia media = +0.0062  IC95% bootstrap [-0.0280, +0.0400] | BM25 mejor en 19, TF-IDF mejor en 15, empates 266 | prueba de signos p = 0.61
  NDCG@10  diferencia media = +0.0366  IC95% bootstrap [+0.0162, +0.0575] | BM25 mejor en 66, TF-IDF mejor en 33, empates 201 | prueba de signos p = 0.0012

Guardado: resultados/metricas_test.json y resultados/metricas_por_consulta.csv


**Lectura de los resultados en `test` (300 consultas).**

| Modelo | P@10 | R@10 | NDCG@10 |
| --- | --- | --- | --- |
| TF-IDF (coseno) | 0.0903 | 0.8183 | 0.6450 |
| BM25 | 0.0910 | 0.8245 | **0.6816** |

* **NDCG@10:** BM25 supera a TF-IDF en +0.0366 (IC 95 % bootstrap [+0.016, +0.058]); es mejor en 66 consultas, peor en 33 y empata en 201; la prueba de signos da p = 0.0012. La diferencia es modesta pero distinguible del azar.
* **P@10 y R@10:** prácticamente iguales (0.0910 frente a 0.0903 y 0.8245 frente a 0.8183) y sin diferencia significativa (p = 0.61): solo cambian en 34 consultas y en las otras 266 ambos modelos empatan.
* **Por qué P@10 es tan bajo:** los qrels de SciFact tienen en promedio 1.13 documentos relevantes por consulta (339 juicios para 300 consultas), de modo que incluso un ranking perfecto tendría P@10 ≈ 0.11. Con tan pocos relevantes la P@10 apenas discrimina; R@10 indica que en promedio los modelos encuentran el 82 % de los relevantes dentro del top 10, y NDCG@10, que premia la posición, es donde los modelos se separan.
* **Contexto externo (no medido aquí):** BEIR publica para BM25 en SciFact un NDCG@10 en torno a 0.665. El 0.6816 obtenido está en ese orden de magnitud; no es una comparación estricta porque el preprocesamiento y la variante de BM25 son distintos.

### 7. Análisis cualitativo

Para entender las cifras se miran consultas concretas: las dos donde BM25 supera más a TF-IDF, las dos donde TF-IDF supera más a BM25 y consultas donde ningún modelo recupera ningún relevante en el top 10. Para cada una se muestran los documentos relevantes y la posición en la que cada modelo los coloca.

In [18]:
def mostrar_consulta(qid):
    rel = {d for d, g in qrels_test[qid].items() if g > 0}
    print(f"\nConsulta {qid}: {texto_consulta[qid]}")
    print("  Relevantes:", [(d, titulos[d][:60]) for d in sorted(rel)][:3], "..." if len(rel) > 3 else "")
    for modelo in MODELOS:
        r = buscador.buscar(texto_consulta[qid], modelo, 10)
        pos = [i + 1 for i, (d, _) in enumerate(r) if d in rel]
        print(f"  {modelo:5} posiciones de relevantes en el top 10: {pos or 'ninguna'} | top 1: {titulos[r[0][0]][:60]!r}")


dif_ndcg = (ancho["ndcg_at_10"]["bm25"] - ancho["ndcg_at_10"]["tfidf"]).sort_values()
print("=== BM25 gana claramente ===")
for qid in dif_ndcg.index[::-1][:2]:
    mostrar_consulta(qid)
print("\n=== TF-IDF gana ===")
for qid in dif_ndcg.index[:2]:
    mostrar_consulta(qid)
ceros = ancho.index[(ancho["r_at_10"]["bm25"] == 0) & (ancho["r_at_10"]["tfidf"] == 0)]
print(f"\n=== Consultas sin ningún relevante en el top 10 con ambos modelos: {len(ceros)} de {len(qids_test)} ===")
for qid in list(ceros)[:3]:
    mostrar_consulta(qid)

=== BM25 gana claramente ===

Consulta 569: In adult tissue, most T cells are memory T cells.
  Relevantes: [('23460562', 'Early life compartmentalization of human T cell differentiat')] 
  bm25  posiciones de relevantes en el top 10: [1] | top 1: 'Early life compartmentalization of human T cell differentiat'
  tfidf posiciones de relevantes en el top 10: [7] | top 1: 'Tissue-resident memory T cells: local specialists in immune '

Consulta 1245: The one-child policy has been successful in lowering population growth.
  Relevantes: [('7662395', 'Perinatal mortality in rural China: retrospective cohort stu')] 
  bm25  posiciones de relevantes en el top 10: [2] | top 1: 'Update on technical issues concerning complementary feeding '
  tfidf posiciones de relevantes en el top 10: ninguna | top 1: 'Reforming the health sector in developing countries: the cen'

=== TF-IDF gana ===

Consulta 133: Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate an

**Comprobación de una hipótesis.** En la consulta 1088 («Silencing of Bcl2…») ningún modelo encuentra el documento relevante, que habla de «BCL-2». La tokenización separa el guion y descarta los tokens de un carácter, de modo que `BCL-2` queda como `bcl` y la consulta tiene `bcl2`: se verifica si ese es el motivo.

In [19]:
doc_rel = corpus.set_index("_id").loc["37549932"]
texto_rel = f"{doc_rel['title']} {doc_rel['text']}"
print("Menciones en el documento relevante:", sorted(set(re.findall(r"(?i)bcl-?2", texto_rel))))
print("Tokens de la consulta   :", preprocesar(texto_consulta["1088"], STOP))
tokens_rel = preprocesar(texto_rel, STOP)
print("'bcl2' en el documento  :", "bcl2" in tokens_rel, "| 'bcl' en el documento:", "bcl" in tokens_rel)

Menciones en el documento relevante: ['BCL-2']
Tokens de la consulta   : ['silenc', 'bcl2', 'import', 'mainten', 'progress', 'tumor']
'bcl2' en el documento  : False | 'bcl' en el documento: True


**Qué muestran los ejemplos.**

* **BM25 supera a TF-IDF (consultas 569 y 1245).** Coloca el documento relevante en la posición 1 (frente a la 7) y en la 2 (frente a ninguna posición del top 10). *Hipótesis, no comprobada aquí:* con consultas cortas formadas por palabras frecuentes del corpus («cells», «tissue», «policy»), el idf y la saturación de la frecuencia de BM25 reparten mejor el peso entre términos que el coseno.
* **TF-IDF supera a BM25 (consultas 133 y 388).** En la 133, larga y llena de términos raros (invadopodia, phosphatidylinositol, Src), TF-IDF pone 4 relevantes en el top 10 y BM25 solo 1. *Hipótesis:* el coseno premia a los documentos que cubren a la vez muchos términos específicos de la consulta, mientras que BM25, que suma contribuciones, puede subir un documento que repite uno o dos de ellos. Ningún modelo gana siempre (66 frente a 33 consultas).
* **Consultas sin ningún relevante en el top 10 con ambos modelos: 35 de 300 (11.7 %).** Los tres ejemplos fallan por motivos distintos:
  * *Consulta 1:* el título del documento relevante no comparte vocabulario evidente con la consulta (hipótesis: diferencia de vocabulario, que un modelo puramente léxico no puede salvar).
  * *Consulta 1049:* el primer resultado de ambos modelos («Growth control and ribosomopathies») es del mismo tema pero no es el documento juzgado relevante. Con apenas ~1 relevante por consulta, los documentos útiles sin juicio cuentan como error (inferencia sobre los qrels, no comprobada).
  * *Consulta 1088:* **comprobado**: el documento relevante escribe «BCL-2» y la consulta «Bcl2»; el preprocesamiento convierte el primero en `bcl` (el guion separa y el `2` suelto se descarta) y el segundo en `bcl2`, que no coinciden (`bcl2` no está en el documento, `bcl` sí). Es una limitación real de la tokenización. No se modificó para no ajustar el sistema mirando `test`; queda como mejora.

### 8. Servicio web (FastAPI)

El servicio es una carpeta autónoma, `servicio/`, que contiene los módulos ya escritos (`preprocesamiento.py`, `indice.py`, `modelos.py`), el corpus (`corpus.jsonl`, copia de los 5 183 documentos para no depender de ninguna descarga), las stopwords, la configuración elegida con `train` y los archivos de despliegue (`vercel.json`, `.python-version`). Al arrancar construye el índice en memoria (unos segundos). Contrato:

* `GET /health` → `{"status": "ok", "documents": N, "models": ["bm25", "tfidf"]}`.
* `POST /search` con `{"query": "...", "model": "bm25" | "tfidf", "top_k": 10}` → `{"model": ..., "results": [{"rank", "document_id", "score", "title", "snippet"}]}`. `model` por defecto `bm25` (acepta mayúsculas), `top_k` entre 1 y 1000 (por defecto 10).
* Resultados ordenados por score descendente (empate: `document_id` ascendente), sin duplicados y exactamente `min(top_k, N)`; el `document_id` es el original del corpus.
* Errores: `422` si falta `query`, está en blanco o solo tiene stopwords, si `model` no es válido o `top_k` está fuera de rango. CORS abierto para clientes web.

In [20]:
%%writefile servicio/app.py
"""Servicio web de recuperación de información sobre SciFact (TF-IDF y BM25)."""
import json
from pathlib import Path
from typing import List, Literal

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field, field_validator

from modelos import MODELOS, Buscador, ConsultaInvalida, fragmento
from preprocesamiento import cargar_stopwords

AQUI = Path(__file__).resolve().parent
CONFIG = json.loads((AQUI / "config.json").read_text(encoding="utf-8"))
BUSCADOR = Buscador.desde_jsonl(AQUI / "corpus.jsonl", cargar_stopwords(AQUI / "stopwords_en.txt"),
                                stem=CONFIG["stem"], k1=CONFIG["k1"], b=CONFIG["b"])

app = FastAPI(title="RI SciFact — TF-IDF y BM25", version="1.0.0",
              description="Búsqueda sobre el corpus SciFact (BeIR) con TF-IDF (coseno) y BM25.")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["GET", "POST", "OPTIONS"], allow_headers=["*"])


class Consulta(BaseModel):
    query: str = Field(..., max_length=5000, examples=["0-dimensional biomaterials show inductive properties"])
    model: Literal["bm25", "tfidf"] = Field("bm25", examples=["bm25"])
    top_k: int = Field(10, ge=1, le=1000, examples=[10])

    @field_validator("model", mode="before")
    @classmethod
    def _normalizar_modelo(cls, v):
        return v.strip().lower() if isinstance(v, str) else v


class Resultado(BaseModel):
    rank: int
    document_id: str
    score: float
    title: str
    snippet: str


class Respuesta(BaseModel):
    model: str
    results: List[Resultado]


@app.get("/")
def raiz():
    return {"servicio": "RI SciFact — TF-IDF y BM25", "corpus": "BeIR/scifact", "documentos": BUSCADOR.indice.n_docs,
            "endpoints": {"GET /health": "disponibilidad", "POST /search": "búsqueda", "GET /docs": "documentación interactiva"}}


@app.get("/health")
def health():
    return {"status": "ok", "documents": BUSCADOR.indice.n_docs, "models": list(MODELOS)}


@app.post("/search", response_model=Respuesta)
def buscar(consulta: Consulta):
    texto = consulta.query.strip()
    if not texto:
        raise HTTPException(status_code=422, detail="La consulta está vacía.")
    try:
        pares = BUSCADOR.buscar(texto, consulta.model, consulta.top_k)
    except ConsultaInvalida as error:
        raise HTTPException(status_code=422, detail=str(error))
    resultados = []
    for rank, (doc_id, score) in enumerate(pares, start=1):
        titulo, cuerpo = BUSCADOR.documentos[doc_id]
        resultados.append(Resultado(rank=rank, document_id=doc_id, score=score, title=titulo, snippet=fragmento(cuerpo)))
    return Respuesta(model=consulta.model, results=resultados)

Overwriting servicio/app.py


In [21]:
%%writefile servicio/requirements.txt
fastapi==0.121.2
uvicorn==0.54.0
numpy==2.5.4
nltk==3.10.4
pydantic==2.14.3

Overwriting servicio/requirements.txt


In [22]:
# Configuración de despliegue para Vercel (el servicio se detecta como FastAPI por su app.py)
(SERVICIO / ".python-version").write_text("3.12\n", encoding="utf-8", newline="\n")
(SERVICIO / ".vercelignore").write_text("__pycache__\n*.pyc\n", encoding="utf-8", newline="\n")
vercel_json = {"$schema": "https://openapi.vercel.sh/vercel.json", "functions": {"app.py": {"maxDuration": 60}}}
(SERVICIO / "vercel.json").write_text(json.dumps(vercel_json, indent=2) + "\n", encoding="utf-8", newline="\n")
print((SERVICIO / "vercel.json").read_text())

{
  "$schema": "https://openapi.vercel.sh/vercel.json",
  "functions": {
    "app.py": {
      "maxDuration": 60
    }
  }
}



In [23]:
%%writefile servicio/README.md
# Recuperación de información sobre SciFact

Servicio web (FastAPI) que indexa el corpus SciFact (BeIR) y ofrece dos rankings independientes:
TF-IDF con similitud coseno y BM25, ambos implementados a mano sobre un índice invertido.

* `GET /health` — comprobación de disponibilidad.
* `POST /search` — cuerpo `{"query": "texto", "model": "bm25" | "tfidf", "top_k": 10}`.
* `GET /docs` — documentación interactiva.

Examen de Recuperación de Información (ICCD753). El código y la evaluación están en el cuaderno del examen.
Se despliega en Vercel (carpeta raíz `servicio/`, framework FastAPI, Python 3.12).

Overwriting servicio/README.md


In [24]:
# Se elimina el __pycache__ generado al importar y se lista lo que se publicará
import shutil
shutil.rmtree(SERVICIO / "__pycache__", ignore_errors=True)
for archivo in sorted(SERVICIO.iterdir()):
    print(f"{archivo.name:22} {archivo.stat().st_size / 1024:9.1f} KB")

.python-version              0.0 KB
.vercelignore                0.0 KB
app.py                       2.7 KB
config.json                  0.0 KB
corpus.jsonl              7817.7 KB
indice.py                    3.2 KB
modelos.py                   4.6 KB
preprocesamiento.py          1.2 KB
README.md                    0.6 KB
requirements.txt             0.1 KB
stopwords_en.txt             1.0 KB
vercel.json                  0.1 KB


**Prueba del servicio en local.** Antes de publicarlo se arranca `uvicorn` como proceso aparte (como si fuera el servidor real, desde la carpeta `servicio/` y sin usar nada del cuaderno) y se le hacen solicitudes HTTP con `requests`, igual que haría Postman: disponibilidad, ambos modelos, tamaños de `top_k`, orden, duplicados, errores `422`, una consulta que no está en los ejemplos y que los `document_id` existen en el corpus. Por último se comprueba que los rankings por HTTP son los mismos que los calculados en memoria y que el NDCG@10 medido por HTTP sobre 30 consultas fijas de `test` coincide con el de la evaluación.

In [25]:
def puerto_libre():
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def probar_servicio(base, esperar=90):
    """Batería de pruebas HTTP contra `base`; devuelve el ndcg@10 por modelo sobre las 30 consultas fijas."""
    limite = time.time() + esperar
    while True:                                                    # espera a que el servicio responda
        try:
            if requests.get(f"{base}/health", timeout=10).status_code == 200:
                break
        except requests.RequestException:
            pass
        assert time.time() < limite, "el servicio no arrancó a tiempo"
        time.sleep(2)

    h = requests.get(f"{base}/health", timeout=30).json()
    assert h == {"status": "ok", "documents": len(corpus), "models": ["bm25", "tfidf"]}, h
    print("GET /health ->", h)

    buscar_http = lambda **cuerpo: requests.post(f"{base}/search", json=cuerpo, timeout=60)
    for modelo in ("bm25", "tfidf"):
        for k in (1, 10, 50):
            r = buscar_http(query="vitamin D deficiency and risk of multiple sclerosis", model=modelo, top_k=k)
            assert r.status_code == 200, r.text
            js = r.json()
            assert js["model"] == modelo and len(js["results"]) == k
            assert [x["rank"] for x in js["results"]] == list(range(1, k + 1))
            assert set(js["results"][0]) == {"rank", "document_id", "score", "title", "snippet"}
            ids_http = [x["document_id"] for x in js["results"]]
            assert len(set(ids_http)) == k and set(ids_http) <= ids_corpus
            sc = [x["score"] for x in js["results"]]
            assert all(a >= b for a, b in zip(sc, sc[1:]))
        print(f"POST /search {modelo}: top_k 1/10/50 correctos, ordenados y sin duplicados")
    primero = buscar_http(query="vitamin D deficiency and risk of multiple sclerosis", model="bm25", top_k=1).json()["results"][0]
    print("  primer resultado:", primero["document_id"], "| score", round(primero["score"], 3), "|", primero["title"][:60], "|", primero["snippet"][:80], "…")
    assert buscar_http(query="Vitamin D", top_k=3).json()["model"] == "bm25"          # modelo por defecto
    assert buscar_http(query="vitamin d", model="BM25", top_k=3).status_code == 200  # acepta mayúsculas
    for cuerpo in ({"query": "x", "model": "otro"}, {"query": "   ", "model": "bm25"}, {"query": "the of and", "model": "tfidf"},
                   {"query": "cancer", "model": "bm25", "top_k": 0}, {"query": "cancer", "top_k": 5000}, {"model": "bm25"}):
        assert requests.post(f"{base}/search", json=cuerpo, timeout=30).status_code == 422, cuerpo
    print("Errores 422 correctos (modelo inválido, consulta vacía o solo stopwords, top_k fuera de rango, sin query)")
    nueva = buscar_http(query="effect of green tea consumption on blood pressure in older adults", model="bm25", top_k=5)
    assert nueva.status_code == 200 and len(nueva.json()["results"]) == 5
    print("Consulta nueva (no incluida en los datos) respondida correctamente")

    # Coincidencia con la evaluación en memoria sobre 30 consultas fijas de test
    muestra = list(np.random.default_rng(7).choice(qids_test, 30, replace=False))
    ndcg = {}
    for modelo in ("bm25", "tfidf"):
        valores = []
        for qid in muestra:
            r = buscar_http(query=texto_consulta[qid], model=modelo, top_k=10).json()["results"]
            local = buscador.buscar(texto_consulta[qid], modelo, 10)
            assert [x["document_id"] for x in r] == [d for d, _ in local], (modelo, qid)
            assert all(math.isclose(x["score"], s, rel_tol=1e-9) for x, (_, s) in zip(r, local))
            gan = {d: g for d, g in qrels_test[qid].items() if g > 0}
            valores.append(ndcg_at_k([x["document_id"] for x in r], gan, 10))
        ndcg[modelo] = float(np.mean(valores))
    return ndcg


puerto = puerto_libre()
log = open(RESULTADOS / "servicio_local.log", "w", encoding="utf-8")
proceso = subprocess.Popen([sys.executable, "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", str(puerto)],
                           cwd=SERVICIO, stdout=log, stderr=subprocess.STDOUT)
try:
    ndcg_http_local = probar_servicio(f"http://127.0.0.1:{puerto}")
finally:
    proceso.terminate()
    proceso.wait(timeout=20)
    log.close()

muestra_fija = list(np.random.default_rng(7).choice(qids_test, 30, replace=False))
ndcg_memoria = {m: float(evaluar(buscador, m, {q: qrels_test[q] for q in muestra_fija})["ndcg_at_10"].mean()) for m in MODELOS}
for m in MODELOS:
    assert math.isclose(ndcg_http_local[m], ndcg_memoria[m], rel_tol=1e-9)
print("\nNDCG@10 sobre 30 consultas fijas de test — por HTTP vs. en memoria:")
print({m: (round(ndcg_http_local[m], 4), round(ndcg_memoria[m], 4)) for m in MODELOS})

GET /health -> {'status': 'ok', 'documents': 5183, 'models': ['bm25', 'tfidf']}
POST /search bm25: top_k 1/10/50 correctos, ordenados y sin duplicados
POST /search tfidf: top_k 1/10/50 correctos, ordenados y sin duplicados
  primer resultado: 22843838 | score 20.368 | [Vitamin D and latitude as environmental factors in multiple | Multiple sclerosis (MS) shows a multifold increase in prevalence with an increas …
Errores 422 correctos (modelo inválido, consulta vacía o solo stopwords, top_k fuera de rango, sin query)
Consulta nueva (no incluida en los datos) respondida correctamente



NDCG@10 sobre 30 consultas fijas de test — por HTTP vs. en memoria:
{'bm25': (0.5909, 0.5909), 'tfidf': (0.5914, 0.5914)}


### 9. Despliegue permanente y gratuito (Vercel)

**Plataforma elegida: Vercel, plan Hobby (gratuito, sin tarjeta).** La primera opción fue Hugging Face Spaces, pero al crear el Space Docker en CPU gratuita la plataforma respondió `402 Payment Required` (ahora lo reserva a la suscripción PRO); se descartó para no pagar nada. Motivos para Vercel: es gratis; entrega una URL HTTPS pública y estable; las funciones se arrancan bajo demanda en lugar de «dormirse» como un contenedor gratuito; y el despliegue sale directamente del repositorio de GitHub, de modo que el código que se evalúa es el publicado.

**Cómo está desplegado.** (1) La carpeta `servicio/` está en el repositorio de GitHub del examen; (2) un proyecto de Vercel con *Root Directory* `servicio` y framework FastAPI detecta `app.py`, instala `requirements.txt` (versiones exactas) y usa Python 3.12 (`.python-version`); (3) cada actualización de la rama de producción vuelve a desplegar. El corpus viaja dentro del despliegue (`corpus.jsonl`), así que el servicio no depende de ninguna otra descarga.

**Limitaciones que conviene conocer.** El plan Hobby es de uso no comercial y tiene límites de uso; no hay garantía contractual de disponibilidad. Tras un periodo sin visitas, la primera solicitud paga un arranque en frío (unos segundos, porque el índice se construye en memoria); las siguientes son inmediatas. La URL y el proyecto persisten mientras la cuenta de Vercel y el proyecto no se eliminen.

**Prueba contra la URL pública.** Se repite la misma batería HTTP de la sección 8, ahora contra la dirección de internet del servicio publicado, y se comprueba que el NDCG@10 medido a través de la red sobre las 30 consultas fijas coincide con el calculado en memoria. Con la variable de entorno `RI_SKIP_DEPLOY=1` esta prueba se omite (por ejemplo, para reproducir el cuaderno sin conexión).

In [26]:
URL_PUBLICA = "https://ri-scifact-pasquel.vercel.app"
PROBAR_PUBLICA = os.environ.get("RI_SKIP_DEPLOY") != "1"

if not PROBAR_PUBLICA:
    print("Pruebas públicas omitidas (RI_SKIP_DEPLOY=1). URL publicada:", URL_PUBLICA)
else:
    ndcg_publico = probar_servicio(URL_PUBLICA, esperar=180)
    for m in MODELOS:
        assert math.isclose(ndcg_publico[m], ndcg_memoria[m], rel_tol=1e-9)
    print(f"\nNDCG@10 sobre 30 consultas fijas de test — URL pública vs. memoria: "
          f"{ {m: (round(ndcg_publico[m], 4), round(ndcg_memoria[m], 4)) for m in MODELOS} }")

Pruebas públicas omitidas (RI_SKIP_DEPLOY=1). URL publicada: https://ri-scifact-pasquel.vercel.app


### 10. Conclusiones

**Resultados.** En las 300 consultas de `test`, BM25 obtiene NDCG@10 = 0.6816 frente a 0.6450 de TF-IDF (+0.0366; IC 95 % [+0.016, +0.058]; prueba de signos p = 0.0012), mientras que P@10 (0.0910 frente a 0.0903) y R@10 (0.8245 frente a 0.8183) son casi idénticos y sin diferencia significativa. BM25 es, por tanto, el modelo recomendado y el valor por defecto del servicio; TF-IDF con coseno es un modelo base razonable, algo peor ordenando.

**Decisiones.** Stemming de Porter y `k1 = 1.2`, `b = 0.75`, elegidos con `train` (el stemming aporta ≈ +0.03 de NDCG@10 en `train`; `k1` y `b` casi no importan). El mismo código (`servicio/*.py`) se usa para evaluar y para servir, y las pruebas HTTP confirman que lo servido coincide con lo evaluado.

**Limitaciones.** (1) Los modelos son léxicos: no resuelven sinónimos ni diferencias de vocabulario (35 de 300 consultas sin relevantes en el top 10). (2) La tokenización separa términos como «BCL-2» y pierde coincidencias con «Bcl2». (3) Los qrels son escasos (1.13 relevantes por consulta), lo que limita P@10 y penaliza documentos útiles sin juicio. (4) El plan gratuito de Vercel (Hobby) es de uso no comercial, sin garantía de disponibilidad, y la primera solicitud tras un periodo de inactividad paga un arranque en frío de unos segundos.

**Mejoras posibles.** Normalizar variantes con guion y términos científicos (`bcl-2` ↔ `bcl2`); BM25 por campos (título y resumen); expansión de consultas con retroalimentación de relevancia; y un reordenamiento o recuperación híbrida con embeddings densos para cubrir el vocabulario distinto.

### Entregables

* **URL pública del servicio:** https://ri-scifact-pasquel.vercel.app
* Documentación interactiva: https://ri-scifact-pasquel.vercel.app/docs
* Código: https://github.com/Vladimirjon/IR_IB_PasquelJohann (carpeta `servicio/`)

**Cómo probarlo desde Postman o `curl`**

```bash
curl https://ri-scifact-pasquel.vercel.app/health

curl -X POST https://ri-scifact-pasquel.vercel.app/search \
  -H "Content-Type: application/json" \
  -d '{"query": "0-dimensional biomaterials show inductive properties", "model": "bm25", "top_k": 5}'
```

En Postman: método `POST`, URL `https://ri-scifact-pasquel.vercel.app/search`, *Body → raw → JSON*:

```json
{"query": "texto de la consulta", "model": "tfidf", "top_k": 10}
```

Si el servicio llevara tiempo sin visitas, la primera solicitud puede tardar unos segundos más (arranque en frío); las siguientes son inmediatas.

**Cómo reproducir.** (1) Crear un entorno con Python 3.12 y las librerías de la primera celda; (2) ejecutar el cuaderno de arriba abajo (descarga SciFact la primera vez); (3) para omitir las pruebas contra la URL pública (por ejemplo, sin conexión), definir `RI_SKIP_DEPLOY=1` antes de abrir el cuaderno.